In [ ]:
# 01 · STATE STAGE ACT 단일 정책 CONFIG · GitHub 중단 복구 · Drive 미사용
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_stage_dagger_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'f7af1f95295b5ecb838d7869f1d7af201e4aec5f',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 1000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 10,
    'team': 'my-team',

    # 실행 조건으로 행동 학습 · 빠른 테스트가 0이면 긴 평가 생략
    'action_training_mode': 'prior',
    'repair_iters': 2000,
    'allow_zero_success_evaluation': False,

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','stage_chunk_policy',
             'stage_pick_sampling','stage_pick_train','stage_all_pick_retrain',
             'stage_pick_finetune','stage_pick_diagnose','stage_reference_check',
             'stage_anchor_continue','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


# Medium Stage ACT · 실제 실패 상태 DAgger 교정

PPO와 sparse reward를 사용하지 않습니다. 34.4% Medium 앵커가 simulator에서 실제로 방문한 상태마다 수집 전용 expert의 교정 행동을 저장하고, 그 데이터를 원본 200개 시연 및 기존 성공 recovery와 함께 supervised 학습합니다.

기존 수집은 expert 행동에 작은 노이즈를 넣은 상태만 보았고 실패 rollout은 버렸습니다. 이 노트북은 learned policy가 만든 XY/Z 오차 상태와 실패 직전 상태도 버리지 않습니다. 각 교정 라벨 다음에 실제 실행 행동이 다르면 action chunk를 1 step에서 끊어, 존재하지 않는 미래 정답을 학습하지 않습니다.

앵커의 observation encoder는 동결합니다. action decoder, stage/gate supervisor와 head만 낮은 learning rate로 업데이트합니다. 매 round는 24개 rollout 수집 → 2,000 update → 공식 `eval.py` 8-seed 평가 순서입니다. 새 16개 확인 시드에서 baseline보다 최소 2개 상자를 더 분류해야 최종 제출에 반영합니다. 실패하면 원래 앵커를 유지합니다. 제출물에는 expert 코드나 규칙 기반 제어가 포함되지 않습니다.


In [ ]:
# 06 · 앵커/DAgger 준비 + 공식 eval.py 실행기
import hashlib, json, os, re, shutil, subprocess, sys
from pathlib import Path
from IPython.display import Video, display
from stage_anchor_continue import ANCHORS, package, prepare
from stage_dagger import prepare_dagger, run_dagger_round

LEVEL = 'medium'
DAGGER_ROUNDS = 4
EPISODES_PER_ROUND = 24
TRAIN_ITERS_PER_ROUND = 2000
BETAS = (.50, .30, .15, 0.)
SELECTION_SEEDS = list(range(68000, 68008))
CONFIRM_SEEDS = list(range(69000, 69016))
MIN_EXTRA_SORTED = 2
MAX_STEPS = 200

UPSTREAM = Path(CFG['repo_dir'])
OFFICIAL_DEFAULT = UPSTREAM/'conf/eval/default.yaml'
anchor_exp = prepare(experiment, run_suffix='_anchor_dagger_base_v1')
RUN_DIR = Path(anchor_exp.run_dir)
DAGGER_ROOT = prepare_dagger(anchor_exp, LEVEL, run_suffix='dagger_medium_v1',
    rounds=DAGGER_ROUNDS, episodes_per_round=EPISODES_PER_ROUND,
    train_iters=TRAIN_ITERS_PER_ROUND, lr=1e-5, betas=BETAS)

def sha256(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

def candidate(name, checkpoint=None):
    overrides = {LEVEL:Path(checkpoint)} if checkpoint is not None else None
    result = package(anchor_exp, checkpoint_overrides=overrides,
                     folder_name='dagger_candidates/'+name)
    manifest = json.loads((result/'manifest.json').read_text(encoding='utf-8'))
    if checkpoint is None:
        assert manifest['levels'][LEVEL]['checkpoint_sha256'] == ANCHORS[LEVEL]['sha256']
    return result

def run_official(item, level, label, seeds=None, show_video=False):
    output = RUN_DIR/level/'dagger_official_eval'/label
    output.mkdir(parents=True, exist_ok=True)
    if seeds is None:
        eval_config = OFFICIAL_DEFAULT
    else:
        eval_config = output/'eval_config.yaml'
        eval_config.write_text('eval:\n  n_episodes: '+str(len(seeds))+            '\n  seeds: '+json.dumps(seeds)+'\n', encoding='utf-8')
    checkpoint = item/'checkpoints'/level/'model.pt'
    sidecar = checkpoint.parent/'policy_config.json'
    identity = dict(level=level, label=label, seeds=seeds,
        checkpoint_sha256=sha256(checkpoint), policy_sha256=sha256(sidecar))
    result_path = output/'result.json'
    if result_path.is_file():
        saved = json.loads(result_path.read_text(encoding='utf-8'))
        if saved.get('identity') == identity:
            print(f'[재사용] {label}: {saved["score"]:.3%}')
            return saved
    command = [sys.executable, str(UPSTREAM/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'policy=stage_policy:load_policy',
        'checkpoint='+str(checkpoint), 'eval_config='+str(eval_config),
        'max_episode_steps='+str(MAX_STEPS), 'hydra.run.dir='+str(output)]
    child_env = dict(os.environ)
    child_env['PYTHONPATH'] = str(item)+os.pathsep+str(UPSTREAM)+os.pathsep+child_env.get('PYTHONPATH','')
    for key in list(child_env):
        if key.startswith('MOVEBOXES_SYNC_'):
            child_env.pop(key, None)
    log = output/'official_eval.log'
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=UPSTREAM, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            errors='replace', bufsize=1)
        try:
            for line in process.stdout:
                handle.write(line); handle.flush(); print(line, end='', flush=True)
            code = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try: process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
            process.stdout.close()
    text = log.read_text(encoding='utf-8')
    match = re.search(r'SORT ACCURACY:\s+([0-9.]+)\s*%', text)
    if code or not match:
        raise RuntimeError(f'official eval failed ({code}); 로그: {log}')
    result = dict(identity=identity, score=float(match.group(1))/100,
                  log=str(log), candidate=str(item))
    result_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
    try:
        anchor_exp.sync_level(level)
    except Exception as error:
        print('GitHub 결과 백업 지연; 로컬 결과는 유지합니다:', error)
    if show_video:
        videos = sorted((output/'videos').rglob('*.mp4'), key=lambda p:p.stat().st_mtime)
        if videos:
            display(Video(str(videos[-1]), embed=True, width=900))
    return result


In [ ]:
# 07 · 원본 Medium 앵커를 새 선택 시드에서 먼저 측정
ANCHOR_CHECKPOINT = RUN_DIR/LEVEL/'anchor.pt'
BASELINE_CANDIDATE = candidate('baseline_anchor')
BASELINE_SELECTION = run_official(
    BASELINE_CANDIDATE, LEVEL, 'selection_baseline', SELECTION_SEEDS)
print('Medium 선택 시드 baseline:', f'{BASELINE_SELECTION["score"]:.3%}',
      '· 기존 34.4%는 seeds 63000..63007의 별도 측정값')


In [ ]:
# 08 · 4개 DAgger round · 수집/학습 중단 시 같은 셀로 재개
BEST_CHECKPOINT = ANCHOR_CHECKPOINT
BEST_SCORE = BASELINE_SELECTION['score']
BEST_ROUND = 0
ROUND_RESULTS = []

for round_number in range(1, DAGGER_ROUNDS+1):
    trained = run_dagger_round(anchor_exp, LEVEL, DAGGER_ROOT,
                               round_number, BEST_CHECKPOINT)
    item = candidate(f'round_{round_number:02d}', trained)
    result = run_official(item, LEVEL, f'selection_round_{round_number:02d}',
                          SELECTION_SEEDS, show_video=True)
    result.update(round=round_number, checkpoint=str(trained))
    ROUND_RESULTS.append(result)
    if result['score'] > BEST_SCORE:
        BEST_CHECKPOINT, BEST_SCORE, BEST_ROUND = trained, result['score'], round_number
        print(f'Round {round_number}: 새 공식 최고 {BEST_SCORE:.3%}')
    else:
        print(f'Round {round_number}: {result["score"]:.3%}; '
              f'현재 최고 {BEST_SCORE:.3%} 유지')
    summary = dict(baseline=BASELINE_SELECTION['score'], best_score=BEST_SCORE,
        best_round=BEST_ROUND, results=[dict(round=row['round'], score=row['score'],
        checkpoint=row['checkpoint']) for row in ROUND_RESULTS])
    (DAGGER_ROOT/'official_rounds.json').write_text(
        json.dumps(summary, indent=2), encoding='utf-8')
    try:
        anchor_exp.sync_level(LEVEL)
    except Exception as error:
        print('GitHub round 백업 지연; 로컬 checkpoint는 유지합니다:', error)

print('DAgger 선택 최고:', dict(round=BEST_ROUND, score=BEST_SCORE,
                                  checkpoint=str(BEST_CHECKPOINT)))


In [ ]:
# 09 · 선택에 쓰지 않은 16개 시드에서 baseline과 최고 DAgger 확인
BASELINE_CONFIRM = run_official(
    candidate('confirm_baseline'), LEVEL, 'confirm_baseline', CONFIRM_SEEDS)

if BEST_ROUND == 0:
    DAGGER_CONFIRM = BASELINE_CONFIRM
    EXTRA_SORTED = 0
    ACCEPT_DAGGER = False
else:
    DAGGER_CONFIRM = run_official(
        candidate('confirm_dagger', BEST_CHECKPOINT), LEVEL,
        'confirm_dagger', CONFIRM_SEEDS, show_video=True)
    EXTRA_SORTED = round((DAGGER_CONFIRM['score']-BASELINE_CONFIRM['score'])*len(CONFIRM_SEEDS)*4)
    ACCEPT_DAGGER = EXTRA_SORTED >= MIN_EXTRA_SORTED

confirmation = dict(seeds=CONFIRM_SEEDS, best_round=BEST_ROUND,
    baseline_score=BASELINE_CONFIRM['score'], dagger_score=DAGGER_CONFIRM['score'],
    extra_sorted=EXTRA_SORTED, required_extra_sorted=MIN_EXTRA_SORTED,
    accepted=ACCEPT_DAGGER)
(DAGGER_ROOT/'confirmation.json').write_text(json.dumps(confirmation, indent=2), encoding='utf-8')
print(json.dumps(confirmation, indent=2))


In [ ]:
# 10 · 확인을 통과한 학습 정책만 제출 반영 · 전체 난이도 평가 · ZIP
overrides = {LEVEL:BEST_CHECKPOINT} if ACCEPT_DAGGER else None
FINAL = package(anchor_exp, checkpoint_overrides=overrides,
                folder_name='final_validated_dagger_candidate')
manifest_path = FINAL/'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if ACCEPT_DAGGER:
    manifest['levels'][LEVEL]['selection'] = 'supervised_dagger'
    manifest['levels'][LEVEL]['dagger_round'] = BEST_ROUND
else:
    assert manifest['levels'][LEVEL]['checkpoint_sha256'] == ANCHORS[LEVEL]['sha256']
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

FINAL_RESULTS = {level:run_official(FINAL, level, 'final_default_'+level,
                                     show_video=(level == LEVEL))
                 for level in ('easy','medium','hard')}
print(json.dumps(dict(dagger_accepted=ACCEPT_DAGGER, best_round=BEST_ROUND,
    final_default_scores={key:value['score'] for key,value in FINAL_RESULTS.items()}), indent=2))
archive = shutil.make_archive(str(RUN_DIR/'stage_act_validated_dagger_submission'),
                              'zip', root_dir=FINAL)
from google.colab import files
print('제출 ZIP:', archive)
files.download(archive)
